# Engine Geometry and Kinematics

This notebook explores the slider-crank mechanism and how it determines cylinder volume, piston motion, and surface areas throughout the engine cycle.

## The Slider-Crank Mechanism

The piston position from TDC is determined by the crank radius `a`, connecting rod length `l`, and crank angle `theta`:

$$x(\theta) = a \cos\theta + \sqrt{l^2 - a^2 \sin^2\theta}$$

The cylinder volume is:

$$V(\theta) = V_c + A_b \cdot (l + a - x(\theta))$$

where $V_c$ is the clearance volume and $A_b$ is the bore area.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from engine_sim.engine.geometry import GeometryParams

## 1. Default Engine Geometry

Let's start with the default engine parameters (86 mm bore/stroke, CR=12.5).

In [ ]:
geom = GeometryParams(bore=0.086, stroke=0.086, con_rod=0.1455, comp_ratio=12.5)

print(f"Bore:             {geom.bore*1000:.1f} mm")
print(f"Stroke:           {geom.stroke*1000:.1f} mm")
print(f"Con Rod:          {geom.con_rod*1000:.1f} mm")
print(f"Crank Radius:     {geom.crank_radius*1000:.1f} mm")
print(f"Rod/Crank Ratio:  {geom.con_rod/geom.crank_radius:.2f}")
print(f"Bore Area:        {geom.bore_area*1e4:.2f} cm2")
print(f"Displacement:     {geom.displacement*1e6:.1f} cm3  ({geom.displacement*1e3:.3f} L)")
print(f"Clearance Volume: {geom.clearance_volume*1e6:.2f} cm3")
print(f"Compression Ratio: {geom.comp_ratio}")

## 2. Piston Position and Cylinder Volume

In [ ]:
theta = np.linspace(-np.pi, np.pi, 500)
ca_deg = np.rad2deg(theta)

# Calculate piston position and volume at each angle
piston_pos = np.array([geom.piston_position(t) for t in theta])
volume = np.array([geom.cylinder_volume(t) for t in theta])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Piston displacement from TDC
x_tdc = geom.piston_position(0.0)
displacement_from_tdc = (x_tdc - piston_pos) * 1000  # mm
ax1.plot(ca_deg, displacement_from_tdc, 'b-', linewidth=2)
ax1.set_ylabel('Displacement from TDC [mm]')
ax1.set_title('Piston Motion')
ax1.axvline(0, color='r', linestyle='--', alpha=0.5, label='TDC')
ax1.axhline(geom.stroke*1000, color='gray', linestyle=':', alpha=0.5, label=f'Stroke = {geom.stroke*1000:.1f} mm')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Cylinder volume
ax2.plot(ca_deg, volume * 1e6, 'g-', linewidth=2)
ax2.axhline(geom.clearance_volume * 1e6, color='r', linestyle=':', alpha=0.5, label=f'Clearance = {geom.clearance_volume*1e6:.1f} cm3')
ax2.axhline((geom.clearance_volume + geom.displacement) * 1e6, color='b', linestyle=':', alpha=0.5,
            label=f'Total = {(geom.clearance_volume + geom.displacement)*1e6:.1f} cm3')
ax2.set_xlabel('Crank Angle [deg]')
ax2.set_ylabel('Volume [cm3]')
ax2.set_title('Cylinder Volume')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Verify compression ratio
V_tdc = geom.cylinder_volume(0)
V_bdc = geom.cylinder_volume(np.pi)
print(f"\nV_TDC = {V_tdc*1e6:.2f} cm3")
print(f"V_BDC = {V_bdc*1e6:.2f} cm3")
print(f"V_BDC / V_TDC = {V_bdc/V_tdc:.2f}  (should equal CR = {geom.comp_ratio})")

## 3. Effect of Compression Ratio

Higher compression ratio means smaller clearance volume relative to displacement.

In [ ]:
cr_values = [8, 10, 12.5, 15, 18]

fig, ax = plt.subplots(figsize=(10, 6))

for cr in cr_values:
    g = GeometryParams(bore=0.086, stroke=0.086, con_rod=0.1455, comp_ratio=cr)
    vol = np.array([g.cylinder_volume(t) for t in theta])
    ax.plot(ca_deg, vol * 1e6, linewidth=2, label=f'CR = {cr}')

ax.set_xlabel('Crank Angle [deg]')
ax.set_ylabel('Volume [cm3]')
ax.set_title('Cylinder Volume vs Compression Ratio')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Volume Rate (dV/dt) and Surface Areas

The volume rate determines the work done by pressure forces ($P \cdot dV/dt$). The surface areas determine heat transfer.

In [ ]:
rpm = 2000.0
dVdt = np.array([geom.volume_rate(t, rpm) for t in theta])
areas = np.array([geom.surface_area(t) for t in theta])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Volume rate
ax1.plot(ca_deg, dVdt * 1e6, 'b-', linewidth=2)
ax1.set_ylabel('dV/dt [cm3/s]')
ax1.set_title(f'Rate of Volume Change (RPM = {rpm:.0f})')
ax1.axhline(0, color='gray', linestyle='-', alpha=0.3)
ax1.axvline(0, color='r', linestyle='--', alpha=0.5, label='TDC')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Surface areas
head_area = areas[:, 0] * 1e4  # cm2
piston_area = areas[:, 1] * 1e4
liner_area = areas[:, 2] * 1e4
total_area = head_area + piston_area + liner_area

ax2.plot(ca_deg, head_area, label='Head', linewidth=2)
ax2.plot(ca_deg, piston_area, label='Piston', linewidth=2, linestyle='--')
ax2.plot(ca_deg, liner_area, label='Liner', linewidth=2)
ax2.plot(ca_deg, total_area, label='Total', linewidth=2, color='black')
ax2.set_xlabel('Crank Angle [deg]')
ax2.set_ylabel('Surface Area [cm2]')
ax2.set_title('Cylinder Surface Areas')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"At TDC: total area = {total_area[len(theta)//2]:.1f} cm2 (liner is minimal)")
print(f"At BDC: total area = {total_area[0]:.1f} cm2 (liner dominates)")

## 5. Comparing Different Engine Sizes

Let's compare a small motorcycle engine, a car engine, and a larger diesel.

In [ ]:
engines = {
    'Motorcycle (250cc)':  GeometryParams(bore=0.072, stroke=0.062, con_rod=0.110, comp_ratio=11.0),
    'HCCI (500cc)':        GeometryParams(bore=0.086, stroke=0.086, con_rod=0.1455, comp_ratio=12.5),
    'Diesel (700cc)':      GeometryParams(bore=0.096, stroke=0.096, con_rod=0.160, comp_ratio=18.0),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for name, g in engines.items():
    vol = np.array([g.cylinder_volume(t) for t in theta])
    ax1.plot(ca_deg, vol * 1e6, linewidth=2, label=f'{name} ({g.displacement*1e6:.0f}cc)')

    # Log-scale P-V for polytropic compression (no combustion)
    P_ivc = 1e5  # 1 bar
    gamma = 1.35
    V_ivc = g.cylinder_volume(-np.pi)  # BDC
    P_poly = P_ivc * (V_ivc / vol)**gamma
    ax2.plot(vol * 1e6, P_poly / 1e5, linewidth=2, label=name)

ax1.set_xlabel('Crank Angle [deg]')
ax1.set_ylabel('Volume [cm3]')
ax1.set_title('Cylinder Volume')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Volume [cm3]')
ax2.set_ylabel('Pressure [bar]')
ax2.set_title('Motored P-V (polytropic)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()